## Comparision and Analysis 

### Loading the saved testing, training dataset, Scalar model for data transformation and created ML models for comparision. Using plotly for dashboarding 

In [1]:
import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)

In [2]:
import plotly.graph_objs as go
import ipywidgets as widgets
from IPython.display import display

In [3]:
pip install plotly pandas numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import joblib
import plotly.graph_objs as go
import autokeras as ak
from tensorflow.keras.models import load_model

# === Load preprocessed data ===
X_test = np.load("/Users/arpitalonakadi/Documents/GIT Final folders/Petrol price/preprocessed_data/X_test.npy")
y_test = np.load("/Users/arpitalonakadi/Documents/GIT Final folders/Petrol price/preprocessed_data/y_test.npy")

# === Load scalers ===
scalers = joblib.load("/Users/arpitalonakadi/Documents/GIT Final folders/Petrol price/preprocessed_data/scalers.pkl")

# === Actual A1 values (inverse scaled) ===
y_test_a1 = scalers['A1'].inverse_transform(y_test[:, 0].reshape(-1, 1)).ravel()

# === Load LSTM model & predict ===
lstm_model = load_model("/Users/arpitalonakadi/Documents/GIT Final folders/Petrol price/models/lstm_model.keras")
lstm_preds = lstm_model.predict(X_test)
lstm_preds_a1 = scalers['A1'].inverse_transform(lstm_preds[:, 0].reshape(-1, 1)).ravel()

# === Load AutoKeras model & predict ===
autokeras_model = load_model(
    "/Users/arpitalonakadi/Documents/GIT Final folders/Petrol price/auto_model/best_model.keras", 
    custom_objects=ak.CUSTOM_OBJECTS
)
X_test_flat = X_test.reshape((X_test.shape[0], -1))
auto_preds = autokeras_model.predict(X_test_flat)
auto_preds_a1 = scalers['A1'].inverse_transform(auto_preds[:, 0].reshape(-1, 1)).ravel()

# === Load saved rolling ARIMA predictions ===
arima_preds = np.load("/Users/arpitalonakadi/Documents/GIT Final folders/Petrol price/models/arima_preds_A1.npy")

# === Create timeline ===
start_date = pd.to_datetime("2020-01-01")
date_range = pd.date_range(start=start_date, periods=len(y_test_a1), freq='W')

# === Define best model visually ===
best_model_name = "LSTM"  # Change this based on your RMSE results

# === Create Plotly figure ===
fig = go.Figure()

# Actual values (bold black)
fig.add_trace(go.Scatter(
    x=date_range, y=y_test_a1,
    mode='lines', name='Actual A1 Price',
    line=dict(color='black', width=4, dash='solid')
))

# LSTM
fig.add_trace(go.Scatter(
    x=date_range, y=lstm_preds_a1,
    mode='lines', name='LSTM Prediction',
    line=dict(color='firebrick', width=3 if best_model_name == "LSTM" else 2.5, dash='solid')
))

# AutoKeras
fig.add_trace(go.Scatter(
    x=date_range, y=auto_preds_a1,
    mode='lines', name='AutoKeras Prediction',
    line=dict(color='mediumseagreen', width=3 if best_model_name == "AutoKeras" else 2.5, dash='dot')
))

# ARIMA
fig.add_trace(go.Scatter(
    x=date_range, y=arima_preds,
    mode='lines', name='ARIMA (Rolling Forecast)',
    line=dict(color='mediumslateblue', width=3 if best_model_name == "ARIMA" else 2.5, dash='dash')
))

# Layout
fig.update_layout(
    title="📊 A1 Petrol Price Forecasting: Model Comparison",
    xaxis_title="Date",
    yaxis_title="A1 Price (USD)",
    legend_title="Model",
    template="plotly_white",
    hovermode="x unified",
    font=dict(family="Arial", size=14)
)

fig.show()

/Users/arpitalonakadi/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


/Users/arpitalonakadi/Library/Python/3.9/lib/python/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 6 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [3]:
# === Trim y_test to match arima_preds length ===
y_test_arima = y_test_a1[-len(arima_preds):]

# Also trim LSTM & AutoKeras predictions (just for consistency)
lstm_trimmed = lstm_preds_a1[-len(arima_preds):]
auto_trimmed = auto_preds_a1[-len(arima_preds):]

In [4]:
print("🔍 Array Shapes for Comparison\n" + "-"*35)
print(f"y_test_a1 shape (original): {y_test_a1.shape}")
print(f"arima_preds shape         : {arima_preds.shape}")
print(f"y_test_arima shape        : {y_test_arima.shape}")
print(f"lstm_preds_a1 shape       : {lstm_preds_a1.shape}")
print(f"lstm_trimmed shape        : {lstm_trimmed.shape}")
print(f"auto_preds_a1 shape       : {auto_preds_a1.shape}")
print(f"auto_trimmed shape        : {auto_trimmed.shape}")

🔍 Array Shapes for Comparison
-----------------------------------
y_test_a1 shape (original): (241,)
arima_preds shape         : (341,)
y_test_arima shape        : (241,)
lstm_preds_a1 shape       : (241,)
lstm_trimmed shape        : (241,)
auto_preds_a1 shape       : (241,)
auto_trimmed shape        : (241,)


In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import plotly.graph_objs as go

# === Trim y_test to match ARIMA (ARIMA is longer) ===
y_test_arima = y_test_a1[:len(arima_preds)]  # shape = 241
arima_preds_trimmed = arima_preds[:len(y_test_arima)]

# === Also trim LSTM & AutoKeras to same length ===
lstm_trimmed = lstm_preds_a1[:len(y_test_arima)]
auto_trimmed = auto_preds_a1[:len(y_test_arima)]

# === Evaluation Function ===
def evaluate_model(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2 Score": r2_score(y_true, y_pred)
    }

# === Evaluate All Models ===
metrics = {
    "LSTM": evaluate_model(y_test_arima, lstm_trimmed),
    "AutoKeras": evaluate_model(y_test_arima, auto_trimmed),
    "ARIMA (Rolling Forecast)": evaluate_model(y_test_arima, arima_preds_trimmed)
}

# === Convert to DataFrame ===
metrics_df = pd.DataFrame(metrics).T.round(4)
print("\n📊 Final Evaluation Metrics:")
display(metrics_df)

# === Plot as Interactive Table (Optional) ===
table_fig = go.Figure(data=[go.Table(
    header=dict(values=["Model", "RMSE", "MAE", "R² Score"],
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[
        metrics_df.index,
        metrics_df["RMSE"],
        metrics_df["MAE"],
        metrics_df["R2 Score"]
    ],
    fill_color='lavender',
    align='left'))
])

table_fig.update_layout(title="📊 Model Evaluation Metrics Comparison")
table_fig.show()

# === Residual Plot ===
residuals_fig = go.Figure()

residuals_fig.add_trace(go.Scatter(
    x=date_range[:len(y_test_arima)],
    y=y_test_arima - lstm_trimmed,
    mode='lines',
    name='LSTM Residuals'
))

residuals_fig.add_trace(go.Scatter(
    x=date_range[:len(y_test_arima)],
    y=y_test_arima - auto_trimmed,
    mode='lines',
    name='AutoKeras Residuals'
))

residuals_fig.add_trace(go.Scatter(
    x=date_range[:len(y_test_arima)],
    y=y_test_arima - arima_preds_trimmed,
    mode='lines',
    name='ARIMA Residuals'
))

residuals_fig.update_layout(
    title="📉 Residuals Over Time (Actual - Predicted)",
    xaxis_title="Date",
    yaxis_title="Prediction Error",
    legend_title="Model",
    template="plotly_white",
    hovermode="x unified"
)

residuals_fig.show()


📊 Final Evaluation Metrics:


,RMSE,MAE,R2 Score
LSTM,0.0776,0.0584,0.9099
AutoKeras,0.1978,0.1381,0.4137
ARIMA (Rolling Forecast),0.5388,0.4346,-3.3491


In [6]:
import plotly.graph_objs as go

metrics_fig = go.Figure(data=[go.Table(
    header=dict(values=["Model", "RMSE", "MAE", "R² Score"],
                fill_color='lightcyan',
                align='left'),
    cells=dict(values=[
        metrics_df.index,  # Model names
        metrics_df["RMSE"],
        metrics_df["MAE"],
        metrics_df["R2 Score"]
    ],
    fill_color='lavender',
    align='left'))
])

In [11]:
import dash
from dash import dcc, html
import plotly.graph_objs as go

# Initialize the Dash app
app = dash.Dash(__name__)
app.title = "Petrol Price Forecasting Dashboard"

# Define the layout
app.layout = html.Div([
    html.H1("⛽ A1 Petrol Price Forecasting: Model Comparison",
            style={'textAlign': 'center', 'color': 'white'}),

    # Forecast Line Plot
    html.Div([
        html.H2("📈 Forecast vs Actual", style={'marginTop': '40px', 'color': 'white'}),
        dcc.Graph(figure=fig, id='forecast-plot')
    ]),

    # Evaluation Metrics Table
    html.Div([
        html.H2("📊 Model Evaluation Metrics", style={'marginTop': '40px', 'color': 'white'}),
        dcc.Graph(figure=metrics_fig, id='metrics-table')
    ]),

    # Residuals Plot
    html.Div([
        html.H2("📉 Residuals Over Time", style={'marginTop': '40px', 'color': 'white'}),
        dcc.Graph(figure=residuals_fig, id='residuals-plot')
    ]),

    html.Hr(style={'borderColor': 'white'}),

    html.Footer("Built By Arpita Lonakadi using Plotly Dash",
                style={'textAlign': 'center', 'padding': '20px', 'color': 'white'})
],
style={
    'backgroundColor': '#111111',
    'padding': '20px',
    'fontFamily': 'Arial, sans-serif'
})

# Run The Dashboard

In [12]:
if __name__ == '__main__':
    app.run(debug=True, port=8050)